# 🏆 iRage AlgoArena — Short-Horizon Return Prediction | v5 FINAL

```
ONE SUBMISSION ONLY — BUILT FOR PRIVATE LEADERBOARD GENERALIZATION
```

## v5 Changes vs v4 (Quant-Reasoned Upgrades)

| Change | v4 | v5 | Why |
|--------|----|----|-----|
| CV Strategy | KFold(shuffle=True) | GroupKFold on CV_GROUP | Prevents temporal leakage across regimes |
| XGBoost | Included (R²=0.0003) | **DROPPED** | Adding noise; optimized weights → 100% CAT anyway |
| CatBoost iters | 5000 (still learning!) | **8000** | bestIteration=4999 every fold = more signal available |
| LGB LR | 0.02 / 12000 rounds | 0.025 / 10000 rounds | Speed-perf tradeoff within 30min Kaggle limit |
| LGB min_child | 300 | **400** | Reduce overfit gap (was 0.018 → target <0.01) |
| LGB lambda_l2 | 8.0 | **12.0** | Same: tighten generalization |
| CatBoost l2_leaf_reg | 10.0 | **15.0** | CatBoost also showed mild overfit |
| Target clip | P2/P98 | **P1/P99** | CatBoost can handle more of the tail signal |
| New FE | 351 features | +18 more | SO3_T cubic, cross-family Lag1 correlation proxy, abs-momentum |
| Ensemble | LGB+XGB+CAT | **LGB+CAT only** | Cleaner signal, optimized weights |

## Evaluation Formula (Final Score)
```
Final Score = 2 × mean_batch_R² − full_test_R²
            = Generalization − Structural Leakage
```
**Strategy: maximize consistency across randomized batches, not full-test R²**
- Avoid any test-set cross-sample operations (no ranking, no KNN, no clustering)
- Global test stats (mean/std) are allowed — we use them for feature scaling sanity only


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — IMPORTS                                           ║
# ╚══════════════════════════════════════════════════════════════╝
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost ✅')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed — LGB only')

SEED = 23
np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
print(f'LightGBM {lgb.__version__}')
print('Imports ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — CONFIG                                            ║
# ╚══════════════════════════════════════════════════════════════╝
#
# v5 PHILOSOPHY:
#   - CatBoost is the primary model (0.055 vs LGB 0.030 in v4)
#   - XGB dropped (0.0003 — pure noise)
#   - GroupKFold on CV_GROUP for regime-aware validation
#   - All regularization tightened to shrink overfit gap
#   - Target clip widened slightly (P1/P99) because CatBoost
#     handles fat tails better than LGB
#
# KAGGLE CONSTRAINTS:
#   - CPU only, 30 minutes total runtime
#   - No internet, no installs, no external data
#   - Must use /kaggle/input/... paths

class CFG:
    # ── Paths (Kaggle standard) ─────────────────────────────────
    DATA_DIR   = Path('/kaggle/input/short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    OUTPUT_DIR = Path('/kaggle/working')

    # ── CV ─────────────────────────────────────────────────────
    # GroupKFold on CV_GROUP = proper regime-aware validation.
    # Each CV_GROUP = independent market regime (not temporal blocks).
    # This prevents the model from learning cross-group leakage.
    N_FOLDS = 5
    SEED    = 23

    # ── Target clipping ────────────────────────────────────────
    # P1/P99: Slightly wider than v4's P2/P98.
    # Rationale: CatBoost with oblivious trees handles tails better.
    # The signal in the 1-2% tail is real momentum, not pure noise.
    # Clipping only affects TRAINING. Test predictions are unconstrained.
    TARGET_CLIP = True
    CLIP_LOWER  = 1    # lower percentile
    CLIP_UPPER  = 99   # upper percentile

    # ── Feature Engineering ────────────────────────────────────
    BUILD_MOMENTUM      = True   # Acc12, Acc23, Div13
    BUILD_ABS_MOMENTUM  = True   # |LagT1|, |LagT2| — regime strength
    BUILD_FAMILY_AGG    = True   # Cross-instrument regime signals
    BUILD_PRICE_FEATS   = True   # Price is the target driver
    BUILD_SO3_POLY      = True   # SO3_T: quad + cubic (session-time effect)

    # ── LightGBM ───────────────────────────────────────────────
    # v5 changes:
    #   min_child_samples: 300→400 (reduce overfit gap from 0.018)
    #   lambda_l2: 8→12 (same intent)
    #   LR: 0.02→0.025, rounds: 12000→10000 (faster, same budget)
    #   early_stop: 300 (generous — low signal regime needs patience)
    LGB_PARAMS = {
        'objective':         'regression',
        'metric':            'rmse',
        'learning_rate':     0.025,
        'num_leaves':        31,
        'max_depth':         -1,
        'min_child_samples': 400,
        'feature_fraction':  0.5,
        'bagging_fraction':  0.7,
        'bagging_freq':      1,
        'lambda_l1':         1.0,
        'lambda_l2':         12.0,
        'min_gain_to_split': 0.01,
        'path_smoothing':    0.1,   # NEW: smooths leaf values along tree path
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }
    LGB_NUM_ROUNDS = 10000
    LGB_EARLY_STOP = 300

    # ── CatBoost ───────────────────────────────────────────────
    # v5 changes:
    #   iterations: 5000→8000 (was still learning at 5000 every fold!)
    #   l2_leaf_reg: 10→15 (reduce mild overfit)
    #   min_data_in_leaf: 200→300 (match LGB tightening)
    #   od_wait: 150→200 (more patience for low-signal regime)
    #   border_count: 128→254 (more split candidates = better splits)
    CAT_PARAMS = dict(
        iterations=8000,
        learning_rate=0.02,
        depth=5,               # depth=4→5: more expressive given 800+ features
        l2_leaf_reg=15.0,
        min_data_in_leaf=300,
        border_count=254,      # NEW: max histogram bins (default 128)
        random_seed=SEED,
        eval_metric='RMSE',
        od_type='Iter',
        od_wait=200,
        verbose=500,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

print('Config ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA LOADING                                      ║
# ╚══════════════════════════════════════════════════════════════╝
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'NaN — Train: {train.isnull().sum().sum():,} | Test: {test.isnull().sum().sum():,}')

# Stash IDs, CV_GROUP, and target BEFORE any modification
train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

# ── CV_GROUP handling ───────────────────────────────────────────
# CV_GROUP was added to train.parquet only (not in test).
# Use it for GroupKFold to prevent temporal leakage.
# If not present (old data format), fall back to KFold.
if 'CV_GROUP' in train.columns:
    cv_groups = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups = len(np.unique(cv_groups))
    print(f'CV_GROUP found: {n_groups} unique groups → using GroupKFold ✅')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups = None
    USE_GROUP_CV = False
    print('CV_GROUP not found → falling back to KFold (shuffle=True)')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

# Column alignment check
assert list(train.columns) == list(test.columns), \
    f'FATAL: train/test column mismatch! {len(train.columns)} vs {len(test.columns)}'

print(f'Features: {train.shape[1]}')
print(f'Train IDs: {train_ids.min()} – {train_ids.max()}')
print(f'Test IDs:  {test_ids.min()} – {test_ids.max()}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — TARGET ANALYSIS & CLIPPING                        ║
# ╚══════════════════════════════════════════════════════════════╝
#
# v5: Using P1/P99 instead of P2/P98.
# Rationale:
#   - Target kurtosis ≈ 48 (extreme fat tails)
#   - The 1-2% range contains real alpha signals (large momentum events)
#   - CatBoost with symmetric trees handles moderately extreme targets well
#   - P2/P98 was perhaps too aggressive — cutting real signal
#   - Net effect: slightly more variance in y, but better signal for CAT

print('══════════════════════════════════════════════')
print('TARGET DISTRIBUTION (raw)')
print('══════════════════════════════════════════════')
desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'\nSkewness:  {y_raw.skew():.4f}  (>0 = right-tailed)')
print(f'Kurtosis:  {y_raw.kurtosis():.4f}  (fat tails expected for returns)')

if CFG.TARGET_CLIP:
    clip_lo = y_raw.quantile(CFG.CLIP_LOWER / 100)
    clip_hi = y_raw.quantile(CFG.CLIP_UPPER / 100)
    y_series  = y_raw.clip(clip_lo, clip_hi)
    n_clipped = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
    pct_clip  = 100 * n_clipped / len(y_raw)
    print(f'\nClip bounds: [{clip_lo:.6f}, {clip_hi:.6f}]  (P{CFG.CLIP_LOWER}/P{CFG.CLIP_UPPER})')
    print(f'Rows clipped: {n_clipped:,} / {len(y_raw):,} ({pct_clip:.2f}%)')
    print(f'Std before: {y_raw.std():.6f}  →  after: {y_series.std():.6f}')
    y = y_series.values.astype(np.float32)
else:
    y = y_raw.values.astype(np.float32)

print('\nTarget ready ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FEATURE CATEGORIZATION                            ║
# ╚══════════════════════════════════════════════════════════════╝
all_cols  = train.columns.tolist()

lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    """First prefix segment = feature family (S01, S02, SO3, etc.)"""
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c + '_LagT1' if (c + '_LagT1') in lag_set else None,
        'lag2': c + '_LagT2' if (c + '_LagT2') in lag_set else None,
        'lag3': c + '_LagT3' if (c + '_LagT3') in lag_set else None,
    }
    for c in pure_base_cols
}

print(f'Base (incl. special):  {len(base_cols):4d}')
print(f'Pure base:             {len(pure_base_cols):4d}')
print(f'LagT1 features:        {len(lag1_cols):4d}')
print(f'LagT2 features:        {len(lag2_cols):4d}')
print(f'LagT3 features:        {len(lag3_cols):4d}')
print(f'Families found:        {families}')
print(f'Total original feats:  {len(all_cols):4d}')
print('Categorization ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — FEATURE ENGINEERING (v5 enhanced)                 ║
# ╚══════════════════════════════════════════════════════════════╝
#
# QUANT RATIONALE FOR v5 NEW FEATURES:
#
#  |LagT1|, |LagT2| (AbsMom):
#    → Raw LagT1 captures direction. Absolute value captures MAGNITUDE.
#    → High |LagT1| = high momentum regime → stronger continuation signal
#    → Asymmetry: models may not learn |x| from x alone without explicit feature
#
#  SO3_T^3 (cubic session time):
#    → SO3_T is likely a session/time-of-day marker
#    → Intraday returns often show cubic shape: low at mid-session,
#      high at open/close
#    → sq + cubic together approximate flexible polynomial intraday curve
#
#  Family LagT1 max - min (family momentum spread):
#    → Within a family, if instruments disagree (some +, some -),
#      this is a DIVERGENCE signal (potential mean reversion)
#    → If all agree (small spread around same sign),
#      this is a CONVERGENCE signal (momentum continuation)
#
#  Price_LagT1 sign * |Price_LagT1| (already = Price_LagT1, but clipped):
#    → Skip: redundant
#
#  STILL NOT BUILT:
#    ❌ Ratios (division): unstable on anonymous scale
#    ❌ Pairwise cross-family products: too many, noisy at R²≈0.05 level
#    ❌ Test-set rankings: explicitly banned by competition rules

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # ── A. MOMENTUM FEATURES ───────────────────────────────────
    # Acc12 = LagT1 - LagT2 : short-term acceleration
    # Acc23 = LagT2 - LagT3 : medium-term acceleration
    # Div13 = LagT1 - LagT3 : full-span divergence
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for base_col, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                acc12_name = base_col + '_Acc12'
                df[acc12_name] = df[l1] - df[l2]
                new_feats.append(acc12_name)
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                acc23_name = base_col + '_Acc23'
                df[acc23_name] = df[l2] - df[l3]
                new_feats.append(acc23_name)
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                div13_name = base_col + '_Div13'
                df[div13_name] = df[l1] - df[l3]
                new_feats.append(div13_name)
        if verbose:
            print(f'  Momentum (Acc12/Acc23/Div13):   {len(new_feats)-n0}')

    # ── B. ABSOLUTE MOMENTUM (v5 NEW) ──────────────────────────
    # |LagT1| = momentum magnitude (regime strength detector)
    # Models learn direction from LagT1 but need explicit magnitude signal
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for base_col, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                abs_name = base_col + '_AbsMom'
                df[abs_name] = df[l1].abs()
                new_feats.append(abs_name)
        if verbose:
            print(f'  AbsMom (|LagT1|):               {len(new_feats)-n0}')

    # ── C. PRICE-SPECIFIC FEATURES ─────────────────────────────
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        p_l1 = 'Price_LagT1'
        p_l2 = 'Price_LagT2'
        p_l3 = 'Price_LagT3'
        if p_l1 in df.columns and p_l2 in df.columns:
            df['Price_Acc']   = df[p_l1] - df[p_l2]
            new_feats.append('Price_Acc')
        if p_l1 in df.columns and p_l3 in df.columns:
            df['Price_Div13'] = df[p_l1] - df[p_l3]
            new_feats.append('Price_Div13')
        if p_l2 in df.columns and p_l3 in df.columns:
            df['Price_Acc23'] = df[p_l2] - df[p_l3]
            new_feats.append('Price_Acc23')
        # Price absolute momentum
        if p_l1 in df.columns:
            df['Price_AbsMom'] = df[p_l1].abs()
            new_feats.append('Price_AbsMom')
        if verbose:
            print(f'  Price-specific:                 {len(new_feats)-n0}')

    # ── D. FAMILY AGGREGATES ───────────────────────────────────
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fam_base = [c for c in pure_base_cols
                        if get_family(c) == fam and c in df.columns]
            fam_lag1 = [c + '_LagT1' for c in fam_base
                        if c + '_LagT1' in df.columns]
            if len(fam_base) < 2:
                continue
            # Level aggregates
            df[fam + '_BaseMean'] = df[fam_base].mean(axis=1)
            df[fam + '_BaseStd']  = df[fam_base].std(axis=1)
            new_feats += [fam + '_BaseMean', fam + '_BaseStd']
            if len(fam_lag1) >= 2:
                # Momentum aggregates
                fam_mom = df[fam_lag1]
                df[fam + '_MomMean'] = fam_mom.mean(axis=1)
                df[fam + '_MomStd']  = fam_mom.std(axis=1)
                new_feats += [fam + '_MomMean', fam + '_MomStd']
                # v5 NEW: momentum spread (max-min within family)
                # High spread = divergence (mean-reversion signal)
                # Low spread  = convergence (momentum continuation)
                df[fam + '_MomSpread'] = fam_mom.max(axis=1) - fam_mom.min(axis=1)
                new_feats.append(fam + '_MomSpread')
        if verbose:
            print(f'  Family aggregates:              {len(new_feats)-n0}')

    # ── E. SO3_T POLYNOMIAL (v5 enhanced) ─────────────────────
    # SO3_T = known covariate (session time / regime marker)
    # Intraday effects are often nonlinear:
    #   - Open/close have different dynamics vs mid-session
    #   - Quadratic: U-shape (high at extremes)
    #   - Cubic: asymmetric (morning vs afternoon drift)
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq']   = df['SO3_T'] ** 2
        df['SO3_T_cb']   = df['SO3_T'] ** 3    # v5 NEW: cubic
        new_feats += ['SO3_T_sq', 'SO3_T_cb']
        # Interaction with price momentum
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        # v5 NEW: SO3_T × family mean momentum (session × macro momentum)
        for fam in families:
            fname = fam + '_MomMean'
            if fname in df.columns:
                df['SO3T_x_' + fam + '_Mom'] = df['SO3_T'] * df[fname]
                new_feats.append('SO3T_x_' + fam + '_Mom')
        if verbose:
            print(f'  SO3_T polynomial+interactions:  {len(new_feats)-n0}')

    if verbose:
        print(f'  ─────────────────────────────────────')
        print(f'  New features:                   {len(new_feats)}')
        print(f'  Total features after FE:        {df.shape[1]}')
    return df, new_feats


print('Running feature engineering...')
t0 = time.time()
train_fe, new_feats = feature_engineering(
    train, lag_map, families, pure_base_cols, verbose=True)
test_fe,  _         = feature_engineering(
    test,  lag_map, families, pure_base_cols, verbose=False)
print(f'\nDone in {time.time()-t0:.1f}s')
print(f'Train_FE: {train_fe.shape} | Test_FE: {test_fe.shape}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — CLEAN & FINALIZE ARRAYS                           ║
# ╚══════════════════════════════════════════════════════════════╝
# Replace inf/-inf produced by any arithmetic on extreme values
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)

assert list(train_fe.columns) == list(test_fe.columns), \
    'FATAL: Column mismatch between train_fe and test_fe!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)

print(f'Total features:       {n_features:5d}')
nan_tr = train_fe.isnull().sum().sum()
nan_te = test_fe.isnull().sum().sum()
print(f'NaN in train_fe:      {nan_tr:10,}  (LGB/CAT handle NaN natively)')
print(f'NaN in test_fe:       {nan_te:10,}')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)

print(f'\nX:      {X.shape} | dtype: {X.dtype}')
print(f'X_test: {X_test.shape} | dtype: {X_test.dtype}')
print(f'y:      {y.shape} | dtype: {y.dtype}')

del train_fe, test_fe, train, test
gc.collect()
print('Memory freed ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — CV SPLITTER SETUP                                 ║
# ╚══════════════════════════════════════════════════════════════╝
#
# v5 KEY CHANGE: GroupKFold on CV_GROUP
#
# WHY THIS MATTERS:
#   The competition confirmed CV_GROUPs are independent market regimes.
#   With plain KFold(shuffle=True), rows from the same temporal regime
#   can appear in BOTH train and val → inflated OOF R² → overconfident submission.
#
#   GroupKFold ensures entire regimes are held out at once → honest OOF.
#   This also matches the evaluation methodology: "multiple temporally
#   randomized test batches" = the private LB is regime-partitioned.
#
# NOTE: GroupKFold does NOT shuffle within groups, but our CV_GROUPs
# are already independent regimes (per competition host), so this is correct.

if USE_GROUP_CV:
    cv_splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits = list(cv_splitter.split(X, y, groups=cv_groups))
    print(f'Using GroupKFold on CV_GROUP ({n_groups} groups → {CFG.N_FOLDS} folds)')
    # Print group distribution per fold
    for i, (trn_idx, val_idx) in enumerate(splits):
        val_grps = np.unique(cv_groups[val_idx])
        print(f'  Fold {i+1}: val rows={len(val_idx):,}  |  val groups={len(val_grps)}')
else:
    kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splits = list(kf.split(X, y))
    print(f'Using KFold (fallback)')

print('CV splitter ready ✅')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — OVERFIT DIAGNOSTIC (SINGLE FOLD, QUICK CHECK)     ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Run fold 1 only, check train vs val gap before committing to full CV.
# v5 targets gap < 0.01 (vs v4's observed 0.018 with LGB).
# If gap is too high → stop and increase lambda_l2 in CFG.

print('══════════════════════════════════════════════')
print('OVERFIT DIAGNOSTIC (fold 1 of 5, LGB quick check)')
print('══════════════════════════════════════════════')

trn_d, val_d = splits[0]

dtrain_d = lgb.Dataset(X[trn_d], label=y[trn_d],
                        feature_name=feature_cols, free_raw_data=True)
dval_d   = lgb.Dataset(X[val_d], label=y[val_d],
                        reference=dtrain_d, free_raw_data=True)

diag_model = lgb.train(
    CFG.LGB_PARAMS,
    dtrain_d,
    num_boost_round=CFG.LGB_NUM_ROUNDS,
    valid_sets=[dtrain_d, dval_d],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
        lgb.log_evaluation(1000),
    ],
)

best_it = diag_model.best_iteration
train_r2_diag = r2_score(y[trn_d], diag_model.predict(X[trn_d], num_iteration=best_it))
val_r2_diag   = r2_score(y[val_d], diag_model.predict(X[val_d], num_iteration=best_it))
gap           = train_r2_diag - val_r2_diag

print(f'\nBest iteration:  {best_it}')
print(f'Train R²:        {train_r2_diag:.6f}')
print(f'Val   R²:        {val_r2_diag:.6f}')
print(f'Gap (train-val): {gap:.6f}', end='  ')

if gap < 0.005:
    print('✅ EXCELLENT — very low overfit, safe to submit')
elif gap < 0.01:
    print('✅ GOOD — acceptable for one-shot submission')
elif gap < 0.02:
    print('⚠️  MODERATE — consider increasing lambda_l2 before submitting')
else:
    print('❌ HIGH OVERFIT — STOP. Increase lambda_l2 or min_child_samples')
    print('   Do NOT submit with this overfit level!')

del diag_model, dtrain_d, dval_d
gc.collect()
print('\nDiagnostic ✅ — proceeding to full 5-fold CV')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — LIGHTGBM: 5-FOLD CV                              ║
# ╚══════════════════════════════════════════════════════════════╝

oof_lgb      = np.zeros(len(y), dtype=np.float64)
pred_lgb     = np.zeros(len(X_test), dtype=np.float64)
lgb_scores   = []
lgb_iters    = []

print('══════════════════════════════════════════════')
print(f'LightGBM  —  {CFG.N_FOLDS}-Fold CV')
print('══════════════════════════════════════════════')

for fold, (trn_idx, val_idx) in enumerate(splits):
    t0 = time.time()
    X_trn, X_val = X[trn_idx], X[val_idx]
    y_trn, y_val = y[trn_idx], y[val_idx]

    dtrain = lgb.Dataset(X_trn, label=y_trn,
                          feature_name=feature_cols, free_raw_data=True)
    dval   = lgb.Dataset(X_val, label=y_val,
                          reference=dtrain, free_raw_data=True)

    model = lgb.train(
        CFG.LGB_PARAMS,
        dtrain,
        num_boost_round=CFG.LGB_NUM_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
            lgb.log_evaluation(1000),
        ],
    )

    best_it          = model.best_iteration
    lgb_iters.append(best_it)

    val_preds        = model.predict(X_val, num_iteration=best_it)
    oof_lgb[val_idx] = val_preds
    pred_lgb        += model.predict(X_test, num_iteration=best_it) / CFG.N_FOLDS

    fold_r2 = r2_score(y_val, val_preds)
    lgb_scores.append(fold_r2)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS}  |  Val R²: {fold_r2:+.6f}  |  '
          f'iter: {best_it}  |  {time.time()-t0:.0f}s')

    del model, dtrain, dval, X_trn, X_val, y_trn, y_val
    gc.collect()

lgb_oof_r2 = r2_score(y, oof_lgb)
lgb_std    = np.std(lgb_scores)

print(f'\n  LGB OOF R²:    {lgb_oof_r2:+.6f}')
print(f'  Fold std:      {lgb_std:.6f}')
print(f'  Stable?        {"✅ YES" if lgb_std < 0.003 else "⚠️  HIGH VARIANCE"}')
print(f'  Avg best iter: {np.mean(lgb_iters):.0f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — LGB FEATURE IMPORTANCE                           ║
# ╚══════════════════════════════════════════════════════════════╝
# Re-run fold 1 briefly to extract feature importance.
# This is for analysis only — doesn't affect predictions.

# Quick single-fold for importance (reuse fold 1 split)
trn_idx, val_idx = splits[0]
dtrain_fi = lgb.Dataset(X[trn_idx], label=y[trn_idx],
                         feature_name=feature_cols, free_raw_data=True)
dval_fi   = lgb.Dataset(X[val_idx], label=y[val_idx],
                         reference=dtrain_fi, free_raw_data=True)

fi_model = lgb.train(
    CFG.LGB_PARAMS,
    dtrain_fi,
    num_boost_round=max(lgb_iters),
    valid_sets=[dval_fi],
    callbacks=[lgb.log_evaluation(-1)],
)

fi_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': fi_model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

print('═══ TOP 30 MOST IMPORTANT FEATURES (v5) ═══')
print(fi_df.head(30).to_string(index=False))

zero_imp = (fi_df['importance'] == 0).sum()
print(f'\nZero importance: {zero_imp}')

fi_path = CFG.OUTPUT_DIR / 'feature_importance.csv'
fi_df.to_csv(fi_path, index=False)
print(f'Saved → {fi_path} ✅')

del fi_model, dtrain_fi, dval_fi
gc.collect()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — CATBOOST: 5-FOLD CV                              ║
# ╚══════════════════════════════════════════════════════════════╝
#
# v5 changes:
#   - iterations: 5000 → 8000 (was hitting max every fold in v4!)
#   - depth: 4 → 5 (more expressive with 800+ engineered features)
#   - l2_leaf_reg: 10 → 15 (tighter regularization)
#   - min_data_in_leaf: 200 → 300
#   - border_count: 128 → 254 (more histogram bins = finer splits)
#   - od_wait: 150 → 200 (more patience in low-signal regime)
#
# CatBoost with oblivious trees is particularly good for this problem:
#   - Symmetric tree structure acts as strong regularizer
#   - Handles anonymous features without domain knowledge
#   - Ordered boosting reduces train-set overfitting

oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores = []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('══════════════════════════════════════════════')
    print(f'CatBoost  —  {CFG.N_FOLDS}-Fold CV')
    print('══════════════════════════════════════════════')

    for fold, (trn_idx, val_idx) in enumerate(splits):
        t0 = time.time()
        pool_trn = Pool(X[trn_idx], y[trn_idx])
        pool_val = Pool(X[val_idx], y[val_idx])

        model = CatBoostRegressor(**CFG.CAT_PARAMS)
        model.fit(pool_trn, eval_set=pool_val, use_best_model=True)

        val_preds        = model.predict(X[val_idx])
        oof_cat[val_idx] = val_preds
        pred_cat        += model.predict(X_test) / CFG.N_FOLDS

        fold_r2 = r2_score(y[val_idx], val_preds)
        cat_scores.append(fold_r2)
        best_it = model.get_best_iteration()
        print(f'  Fold {fold+1}/{CFG.N_FOLDS}  |  Val R²: {fold_r2:+.6f}  |  '
              f'best_iter: {best_it}  |  {time.time()-t0:.0f}s')

        del model, pool_trn, pool_val
        gc.collect()

    cat_oof_r2   = r2_score(y, oof_cat)
    cat_std      = np.std(cat_scores)
    CATBOOST_DONE = True

    print(f'\n  CAT OOF R²:    {cat_oof_r2:+.6f}')
    print(f'  Fold std:      {cat_std:.6f}')
    print(f'  Stable?        {"✅ YES" if cat_std < 0.003 else "⚠️  HIGH VARIANCE"}')
else:
    cat_oof_r2 = 0.0
    cat_std    = 0.0
    print('CatBoost not available — skipping')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — ENSEMBLE OPTIMIZATION                            ║
# ╚══════════════════════════════════════════════════════════════╝
#
# v5: LGB + CAT ensemble only (XGB dropped — it added noise in v4).
#
# Weight optimization strategy:
#   - Optimize weights to maximize OOF R² (unbiased estimate)
#   - Use Nelder-Mead (gradient-free, robust for 2D)
#   - Constrain weights to [0, 1] via sigmoid transformation
#   - Simple average as fallback if optimization fails
#
# NOTE: This is NOT test-set exploitation.
# Weights are fitted on TRAINING OOF predictions only.

model_names  = ['LGB', 'CAT'] if CATBOOST_DONE else ['LGB']
oofs_list    = [oof_lgb, oof_cat] if CATBOOST_DONE else [oof_lgb]
preds_list   = [pred_lgb, pred_cat] if CATBOOST_DONE else [pred_lgb]

print('══════════════════════════════════════════════')
print('ENSEMBLE OPTIMIZATION (OOF-based)')
print('══════════════════════════════════════════════')

for name, oof in zip(model_names, oofs_list):
    print(f'  {name}  OOF R²: {r2_score(y, oof):+.6f}')

# Simple average baseline
oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'\nSimple avg OOF R²: {r2_sa:+.6f}')

# Optimized weights (only worthwhile if 2+ models)
if len(oofs_list) > 1:
    oofs_arr  = np.stack(oofs_list, axis=1)   # (n_train, n_models)
    preds_arr = np.stack(preds_list, axis=1)  # (n_test, n_models)

    def neg_r2(raw_w):
        # Softmax to ensure positive weights summing to 1
        w = np.exp(raw_w) / np.exp(raw_w).sum()
        blend = (oofs_arr * w).sum(axis=1)
        return -r2_score(y, blend)

    # Multiple restarts for robustness
    best_r2_opt = -np.inf
    best_w      = None
    n_models    = len(oofs_list)

    for init_seed in range(10):
        rng    = np.random.RandomState(init_seed)
        w0     = rng.randn(n_models) * 0.1
        result = minimize(neg_r2, w0, method='Nelder-Mead',
                          options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-8})
        if -result.fun > best_r2_opt:
            best_r2_opt = -result.fun
            best_w      = np.exp(result.x) / np.exp(result.x).sum()

    oof_opt  = (oofs_arr  * best_w).sum(axis=1)
    pred_opt = (preds_arr * best_w).sum(axis=1)
    r2_opt   = r2_score(y, oof_opt)

    print(f'Optimized OOF R²:   {r2_opt:+.6f}')
    for name, w in zip(model_names, best_w):
        print(f'  Weight {name}: {w:.4f}')

    # Use optimized if it improves over simple average
    if r2_opt > r2_sa:
        final_oof  = oof_opt
        final_pred = pred_opt
        final_r2   = r2_opt
        method_str = 'Optimized wts'
        print(f'\n→ Using OPTIMIZED weights (improvement: {r2_opt-r2_sa:+.6f})')
    else:
        final_oof  = oof_sa
        final_pred = pred_sa
        final_r2   = r2_sa
        method_str = 'Simple avg'
        print(f'\n→ Using SIMPLE AVERAGE (optimized did not improve)')
else:
    # Single model
    final_oof  = oofs_list[0]
    final_pred = preds_list[0]
    final_r2   = r2_score(y, final_oof)
    method_str = 'LGB only'
    r2_opt     = final_r2
    print(f'\n→ Single model (LGB): R² = {final_r2:+.6f}')

print(f'\n★ FINAL OOF R²: {final_r2:+.6f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — SUBMISSION GUARDS & SAVE                         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# 6 hard assertions before writing the file.
# Any failure = crash loud. Never write a silent bad submission.

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Build submission ───────────────────────────────────────────
submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

# ── 6 Guards ───────────────────────────────────────────────────
assert len(submission) == len(test_ids),                         'FATAL: Wrong row count!'
assert list(submission.columns) == ['ID', 'TARGET'],            'FATAL: Wrong columns!'
assert submission['ID'].nunique() == len(test_ids),             'FATAL: Duplicate IDs!'
assert not submission['TARGET'].isnull().any(),                  'FATAL: NaN in TARGET!'
assert not np.isinf(submission['TARGET'].values).any(),          'FATAL: Inf in TARGET!'
assert submission['TARGET'].std() > 0,                           'FATAL: Constant predictions!'

print('All 6 submission guards passed ✅')

# ── Save ───────────────────────────────────────────────────────
sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

verify = pd.read_csv(sub_path)
assert len(verify) == len(test_ids),                'FATAL: Saved file has wrong row count!'
assert list(verify.columns) == ['ID', 'TARGET'],    'FATAL: Saved file has wrong columns!'

print(f'\n✅ submission.csv SAVED AND VERIFIED')
print(f'   Path:     {sub_path}')
print(f'   Rows:     {len(submission):,}')
print(f'   ID range: {submission["ID"].min()} – {submission["ID"].max()}')
print()
print('First 10 rows:')
print(submission.head(10).to_string(index=False))
print()
print('TARGET distribution:')
print(submission['TARGET'].describe())

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — FINAL R² SCORECARD                               ║
# ╚══════════════════════════════════════════════════════════════╝

print()
print('╔' + '═'*65 + '╗')
print('║   R² SCORECARD — iRage Short-Horizon Return Prediction (v5)  ║')
print('╠' + '═'*65 + '╣')
print(f'║  Features (original):  {445:<41d} ║')
print(f'║  Features (after FE):  {n_features:<41d} ║')
print(f'║  Train rows:           {len(X):<41,} ║')
print(f'║  Test rows:            {len(X_test):<41,} ║')
print(f'║  CV strategy:          {"GroupKFold(CV_GROUP)" if USE_GROUP_CV else "KFold(shuffle)":<41} ║')
print('╠' + '═'*65 + '╣')
print('║  INDIVIDUAL MODEL OOF R²                                       ║')
for name, oof in zip(model_names, oofs_list):
    r2 = r2_score(y, oof)
    bar_len = max(0, int(r2 * 800))
    bar = '█' * min(bar_len, 40)
    print(f'║  {name:<5}  {r2:+.6f}  {bar:<40} ║')
print('╠' + '═'*65 + '╣')
print('║  ENSEMBLE                                                      ║')
if len(oofs_list) > 1:
    print(f'║  Simple avg:    {r2_sa:+.6f}                                   ║')
    print(f'║  Optimized:     {r2_opt:+.6f}                                   ║')
print(f'║  ★ FINAL ({method_str:<18}): {final_r2:+.6f}               ║')
print('╠' + '═'*65 + '╣')
print('║  OVERFIT CHECK (LGB fold 1)                                    ║')
print(f'║  Train R²: {train_r2_diag:.6f}  |  Val R²: {val_r2_diag:.6f}              ║')
gap_flag = '✅ LOW' if (train_r2_diag-val_r2_diag)<0.01 else '⚠️  CHECK'
print(f'║  Gap: {train_r2_diag-val_r2_diag:.6f} — {gap_flag:<56} ║')
print('╠' + '═'*65 + '╣')
print('║  INTERPRETATION (quant context)                                ║')
if final_r2 > 0.05:
    level = 'EXCEPTIONAL ★★★ — top leaderboard territory'
elif final_r2 > 0.03:
    level = 'STRONG ★★ — competitive, prize range'
elif final_r2 > 0.01:
    level = 'GOOD ★ — above majority of field'
elif final_r2 > 0.005:
    level = 'DECENT — positive signal'
elif final_r2 > 0:
    level = 'WEAK — barely positive'
else:
    level = 'NO SIGNAL — model is predicting noise'
print(f'║  {level:<63} ║')
print('╠' + '═'*65 + '╣')
print('║  FINAL EVALUATION FORMULA (competition)                        ║')
print('║  Final Score = 2 × mean_batch_R² − full_test_R²               ║')
print('║  → Consistent batch-R² > full-test-R² = best outcome          ║')
print('╠' + '═'*65 + '╣')
print('║  OUTPUT FILES                                                  ║')
print('║  /kaggle/working/submission.csv          ← SUBMIT THIS        ║')
print('║  /kaggle/working/feature_importance.csv  ← investigate        ║')
print('╚' + '═'*65 + '╝')
print()
print('✅ NOTEBOOK COMPLETE — v5')
print('   Run top-to-bottom = submission.csv + full R² scorecard')

---
## 📌 ONE-SHOT SUBMISSION CHECKLIST (v5)

Before clicking **Submit Prediction** on Kaggle, verify ALL of these:

- [ ] **LGB fold 1 gap < 0.01** — overfit check passed (v4 was 0.018, v5 targets < 0.01)
- [ ] **CatBoost best_iter < 7800** — if it hits 8000 every fold, increase to 10000
- [ ] **Final OOF R² > 0.03** — model has strong positive signal
- [ ] **Fold std < 0.003** — stable across regime groups
- [ ] **Cell 14 printed `✅ submission.csv SAVED AND VERIFIED`**
- [ ] **submission.csv has 410,139 rows**
- [ ] **Two columns only: ID, TARGET**
- [ ] **No NaN, no Inf in TARGET column**

---
## 📌 v5 Design Notes

### Why GroupKFold wins over KFold here
The competition evaluates on *randomized batch R²*. If the model learned regime-specific patterns from validation leakage (via shuffled KFold), its batch R² will be inconsistent. GroupKFold on CV_GROUP ensures our model was never shown the validation regime during training — exactly matching the evaluation setup.

### Why XGB was dropped
XGB OOF R² in v4: +0.0003. Adding it to the ensemble moves the optimized weights to ~100% CatBoost anyway. The optimizer confirms XGB adds no signal. Removing it: (1) faster, (2) cleaner ensemble, (3) no noise from a bad model.

### Why CatBoost dominates
CatBoost's ordered boosting prevents target leakage within the training fold. On a low-SNR prediction task (return prediction with kurtosis=48), this matters enormously. LGB without ordered boosting can overfit individual extreme observations.

### Final Score formula implications
```
Final Score = 2 × mean_batch_R² − full_test_R²
```
If our full_test R² = 0.055 and mean_batch R² = 0.050, Final = 0.045.
If we overfit to test structure: full = 0.060, batch = 0.040, Final = 0.020.
**Consistent generalization beats test-set exploitation.**

| OOF R² (v5 expected) | Estimated rank |
|---|---|
| > 0.055 | Top 1-3 |
| 0.040–0.055 | Top 3-10 |
| 0.025–0.040 | Top 10-30 |
| 0.010–0.025 | Top 30-60 |